## Car Price Prediction Project

Built a regression model in PySpark to predict car selling price, including feature engineering and model evaluation  

Tools: PySpark, Databricks

### Data loading

In [0]:
# Data Loading
# Loading dataset from Databricks table

df = spark.table("car_data")
df.show()

### Data understanding

In [0]:
# Data Understanding
# Inspecting schema and summary statistics

df.printSchema()
display(df.describe())

### Data cleaning

In [0]:
# Data Cleaning
# Removing missing values

df_clean = df.dropna()

### Feature engineering

In [0]:
# Feature engineering
# Creating new feature: car age

from pyspark.sql.functions import col

df_features = df_clean.withColumn("car_age", 2026 - col("Year"))

### Data analysis

In [0]:
# Data Analysis
# Exploring relationships between variables

from pyspark.sql.functions import avg

# price vs fuel type
df_features.groupBy("Fuel_Type").agg(avg("Selling_Price").alias("avg_selling_price")).show()

# price vs transmission
df_features.groupBy("Transmission").agg(avg("Selling_Price").alias("avg_price")).show()

# price vs age
df_features.groupBy("car_age").agg(avg("Selling_Price").alias("avg_price")).orderBy("car_age").show()
print(f'Correlation between selling price vs age: {df_features.stat.corr("Selling_Price", "car_age")}')

print(f'Correlation between selling price vs present price: {df_features.stat.corr("Selling_Price", "Present_Price")}')

### Feature preparation for ML

In [0]:
# Feature Preparation
# Converting categorical variables into numeric format

from pyspark.ml.feature import StringIndexer

fuel_indexer = StringIndexer(inputCol="Fuel_Type", outputCol="Fuel_Type_idx")
trans_indexer = StringIndexer(inputCol="Transmission", outputCol="Transmission_idx")

df_ml = fuel_indexer.fit(df_features).transform(df_features)
df_ml = trans_indexer.fit(df_ml).transform(df_ml)

### Vector Assembly

In [0]:
# Vector Assembler
# Combining features into a single vector

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["car_age", "Kms_Driven", "Fuel_Type_idx", "Transmission_idx", "Present_Price"],
    outputCol="features"
)

df_ml = assembler.transform(df_ml)

# Improved model performance significantly by adding a highly relevant feature (present price), increasing R² from ~0.4 to ~0.8.

### Train/Test split

In [0]:
# Train/Test Split
# Splitting data into training and testing sets

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

### Linear Regression Model

In [0]:
# Spit dataset for train and test sets
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Model build
# Linear Regression
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="Selling_Price")
lr_model = lr.fit(train_df)# Model Training - Linear Regression

from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="Selling_Price"
)

model = lr.fit(train_df)

### Model evaluation

In [0]:
# Model Evaluation

from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test_df)

evaluator = RegressionEvaluator(
    labelCol="Selling_Price",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)

evaluator_rmse = RegressionEvaluator(
    labelCol="Selling_Price",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

print("R2:", r2)
print("RMSE:", rmse)

### Random Forest Model 

In [0]:
# Model Comparison - Random Forest

from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Selling_Price"
)

rf_model = rf.fit(train_df)

rf_predictions = rf_model.transform(test_df)

r2_rf = evaluator.evaluate(rf_predictions)
rmse_rf = evaluator_rmse.evaluate(rf_predictions)

print("RF R2:", r2_rf)
print("RF RMSE:", rmse_rf)

# Linear Regression outperformed Random Forest due to the relatively small dataset and linear relationships in the data.

### Summary

- Present price is the most important feature for predicting\
 selling price.
- Linear Regression performed better than Random Forest.
- Feature engineering significantly improved model performance.